# Transform Drivers Data

1. Read bronze `drivers` table
1. Keep only the columns required for analytics (Drop `url` column)
1. Standardise column names using snake_case (`driverId` → `driver_id`, `dateOfbirth` → `date_of_birth`)
1. Concatenate `name.givenName` and `name.familyName` to create a new column called `driver_name` and transform the value to Title Case
1. Remove duplicate records
1. Transform values of column `nationality` to Title Case
1. Write the transformed data to silver `drivers` table



#### Entity Relationship Diagram - Formula1 Bronze Schema

![Formula1 Raw Data.png](../../z-course-images/formula1-raw-data-erd.png "Formula1 Raw Data.png")

In [0]:
%run ../../../proyecto_final_formula1/PrepAmb/03.Configuracion_Ambiente

In [0]:
%run ../../../proyecto_final_formula1/PrepAmb/04.Ayuda_Silver

In [0]:
 #%run "../02-Silver/91.Build_Nationality_Region_Reference"


In [0]:
tabla_bronze = f"{catalogo}.{esquema_bronze}.drivers"
tabla_silver= f"{catalogo}.{esquema_silver}.pilotos"
ref_nationality_region_df = spark.table(f"{catalogo}.{esquema_silver}.ref_nacionalidad_region")

In [0]:
from pyspark.sql import functions as F

#### Step 1 - Read bronze `drivers` table

In [0]:
drivers_df = spark.table(tabla_bronze)

#### Step 2 - Keep only the columns required for analytics (Drop url column)

In [0]:
drivers_dropped_df = drivers_df.drop(F.col("url"))

#### Step 3 - Standardise Column Names
- Standardise column names using snake_case (`driverId` → `driver_id`, `dateOfBirth` → `date_of_birth`)

In [0]:
drivers_renamed_df = (
    drivers_dropped_df
        .withColumnsRenamed({
            "driverId": "id_piloto",
            "dateOfBirth": "fecha_nacimiento",
            "nationality": "nacionalidad",
            "number": "numero",
            "ingesta_timestamp": "ingesta_timestamp",
            "archivo_fuente": "archivo_fuente"
        })
)

In [0]:
display(drivers_renamed_df)

#### Step 4 - Concatenate name.givenName and name.familyName to create a new column called driver_name

In [0]:
drivers_concatenated_df = (
  drivers_renamed_df
       .withColumn("nombre_piloto", 
                   F.initcap(F.concat_ws(" ", F.col("name.givenName"), F.col("name.familyName"))))
       .drop("name")
)

In [0]:
display(drivers_concatenated_df)

#### Step 5 - Remove duplicate records

In [0]:
drivers_distinct_df = drivers_concatenated_df.dropDuplicates(["id_piloto"])

In [0]:
display(drivers_distinct_df)

In [0]:
drivers_distinct_df_join = (
    drivers_distinct_df 
        .join(
            ref_nationality_region_df,  # 1. El DataFrame con el que te vas a unir va aquí
            F.initcap(drivers_distinct_df.nacionalidad) == ref_nationality_region_df.nationality, # 2. Condición corregida
            "left"
        ).select(
           drivers_distinct_df.id_piloto,
           drivers_distinct_df.fecha_nacimiento,
            drivers_distinct_df.nombre_piloto,
            drivers_distinct_df.nacionalidad,
            ref_nationality_region_df.region,
             drivers_distinct_df.ingesta_timestamp,
             drivers_distinct_df.archivo_fuente
        )
)

display(drivers_distinct_df_join)

#### Step 6 - Transform values of column `nationality` to Title Case

In [0]:
from pyspark.sql.functions import when, col
df_drivers_final= (

   drivers_distinct_df_join   
         .withColumn("nacionalidad", traducir_f1_udf( F.initcap( F.col("nacionalidad") ) ))
         .withColumn( "region",
             when(col("region") =="Europe",  "Europa")
             . when(col("region") =="North America",  "Norteamérica")
             . when(col("region") =="South America",  "Suramérica")
             . when(col("region") =="Africa",  "Africa")
             .when(col("region") =="Asia",  "Asia")
             .when(col("region") =="Oceania",  "Oceania")
             .otherwise("Desconocido")
))

In [0]:
display(df_drivers_final)

#### Step 7 - Write the transformed data to silver `drivers` table

In [0]:
(
    df_drivers_final
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(tabla_silver)
)

In [0]:
display(spark.table(tabla_silver))